In [ ]:
import time

import numpy as np
import pandas as pd
from ranx import Qrels, Run, evaluate

from common.corpus import load_corpus
from common.goldset import load_queries, qrels
from common import retrieval as R

docs = load_corpus()
chunks = R.chunk_corpus(docs)
queries = load_queries()
gold = Qrels(qrels())

bm25 = R.BM25Retriever(chunks)
dense = R.DenseRetriever(chunks)
print(f"{len(chunks)} Chunks, {len(queries)} Gold-Queries")

In [ ]:
def rrf(rankings: list[list[str]], k: int = 60) -> dict[str, float]:
    fused: dict[str, float] = {}
    for ranking in rankings:
        for rang, cid in enumerate(ranking, start=1):
            fused[cid] = fused.get(cid, 0.0) + 1.0 / (k + rang)
    return fused


by_id = {c.chunk_id: c for c in chunks}


def hybrid_search(query: str, k: int = 10, pool: int = 20):
    bm = [c.chunk_id for c, _ in bm25.search(query, pool)]
    dn = [c.chunk_id for c, _ in dense.search(query, pool)]
    fused = rrf([bm, dn])
    top = sorted(fused.items(), key=lambda x: -x[1])[:k]
    return [(by_id[cid], s) for cid, s in top]

In [ ]:
def run_for(searcher) -> dict:
    out = {}
    for q in queries:
        scored = searcher(q.text)
        out[q.qid] = {d: s for d, s in R.collapse_to_docs(scored)}
    return out


def hybrid_rerank(query: str, k: int = 10, pool: int = 12):
    cand = hybrid_search(query, k=pool)
    return R.cross_encoder_rerank(query, cand, k=k)


setups = {
    "BM25":         lambda q: bm25.search(q, 10),
    "Dense":        lambda q: dense.search(q, 10),
    "Hybrid-RRF":   lambda q: hybrid_search(q, 10),
    "Hybrid+Rerank": lambda q: hybrid_rerank(q, 10),
}

zeilen = []
laeufe = {}
for name, fn in setups.items():
    t0 = time.perf_counter()
    run = run_for(fn)
    dt = time.perf_counter() - t0
    laeufe[name] = run
    m = evaluate(gold, Run(run, name=name), ["ndcg@10", "recall@10", "mrr"])
    zeilen.append({"setup": name, **{k: round(float(v), 3) for k, v in m.items()},
                   "sek/Query": round(dt / len(queries), 3)})

df = pd.DataFrame(zeilen).set_index("setup")
print(df)

In [ ]:
def ndcg_pro_query(run, qid):
    return float(evaluate(Qrels({qid: qrels()[qid]}), Run({qid: run[qid]}), ["ndcg@10"]))


print(f"{'qid':4s} {'BM25':>6s} {'Dense':>6s}  Frage")
for q in queries:
    nb = ndcg_pro_query(laeufe["BM25"], q.qid)
    nd = ndcg_pro_query(laeufe["Dense"], q.qid)
    if abs(nb - nd) > 0.1:
        print(f"{q.qid:4s} {nb:6.2f} {nd:6.2f}  {q.text[:46]}")

In [ ]:
exakt = "Teile-Nummer P12-1001"
print("BM25 :", [c.doc_id for c, _ in bm25.search(exakt, 3)])
print("Dense:", [c.doc_id for c, _ in dense.search(exakt, 3)])

In [ ]:
def hybrid_k(query, kk, pool=20):
    bm = [c.chunk_id for c, _ in bm25.search(query, pool)]
    dn = [c.chunk_id for c, _ in dense.search(query, pool)]
    top = sorted(rrf([bm, dn], k=kk).items(), key=lambda x: -x[1])[:10]
    return [(by_id[cid], s) for cid, s in top]


for kk in (10, 60, 200):
    run = run_for(lambda q, kk=kk: hybrid_k(q, kk))
    ndcg = float(evaluate(gold, Run(run), ["ndcg@10"]))
    print(f"rrf k={kk:3d}: nDCG@10={ndcg:.3f}")